In [1]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB6
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE

2025-06-02 08:32:59.277902: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-02 08:32:59.288094: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748853179.297521   19861 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748853179.300319   19861 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-02 08:32:59.313064: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Preprocessing and segregate

In [2]:
def preprocessing(path):
    img = Image.open(path)
    img_resized = img.resize((224,224))
    return np.array(img_resized)/255

In [3]:
data_path = '../data/spectrograms'
class0 = 'class_0'
class1 = 'class_1'

In [4]:
non_seizure = []
seizure = []
n_seizure_names = [f for f in os.listdir(os.path.join(data_path,class0))]
seizure_names = [f for f in os.listdir(os.path.join(data_path,class1))]
for path in n_seizure_names:
    non_seizure.append(preprocessing(os.path.join(data_path,class0,path)))
for path in seizure_names:
    seizure.append(preprocessing(os.path.join(data_path,class1,path)))

In [5]:
non_seizure_labels = [0]*len(non_seizure)
seizure_labels = [1]*len(seizure)

data = non_seizure+seizure
data_labels = non_seizure_labels+seizure_labels

X, y = shuffle(np.array(data), np.array(data_labels),random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state = 42)
X_test, X_val, y_test, y_val = train_test_split(X_test,y_test,test_size=1/3,random_state = 42)
y_test =np.eye(2)[y_test] 
y_val = np.eye(2)[y_val] 

In [6]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

X_train shape: (280, 224, 224, 4)
y_train shape: (280,)
X_test shape: (80, 224, 224, 4)
y_test shape: (80, 2)
X_val shape: (40, 224, 224, 4)
y_val shape: (40, 2)


## Applying SMOTE

In [7]:
X_flat = X_train.reshape((X_train.shape[0], -1))
smote=SMOTE(sampling_strategy='minority') 
X_train_resampled,y_train_resampled = smote.fit_resample(X_flat,y_train)
X_train = X_train_resampled.reshape((-1, 224, 224, 4))
y_train = np.eye(2)[y_train_resampled]

In [8]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (414, 224, 224, 4)
y_train shape: (414, 2)


## Model Creeation

In [9]:
base_model = EfficientNetB6(
    include_top=False,
    weights=None,
    input_shape=(224, 224, 4),
    pooling='avg'
)

x = base_model.output
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(2, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)


/home/ricky/miniconda3/envs/CUDA/lib/python3.12/site-packages/keras/src/applications/efficientnet.py:289: UserWarning: This model usually expects 1 or 3 input channels. However, it was passed an input_shape with 4 input channels.
  input_shape = imagenet_utils.obtain_input_shape(
I0000 00:00:1748853185.416053   19861 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [10]:
model.compile(optimizer = Adam(), loss = 'categorical_crossentropy', metrics = ['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    '../weights/EfficientNet.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

## Model Training

In [11]:
history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size = 20,
        callbacks= [checkpoint]
    )

Epoch 1/30


I0000 00:00:1748853233.990538   19957 service.cc:148] XLA service 0x7fa4940025d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748853233.991098   19957 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-06-02 08:33:55.578136: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1748853241.542101   19957 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-06-02 08:34:09.245314: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_58463', 16 bytes spill stores, 16 bytes spill loads

2025-06-02 08:34:09.430659: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_584

20/21 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step - accuracy: 0.5656 - loss: 1.1212

2025-06-02 08:35:17.147968: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_58463_0', 256 bytes spill stores, 256 bytes spill loads

2025-06-02 08:35:17.414031: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_62950', 8 bytes spill stores, 8 bytes spill loads

2025-06-02 08:35:29.833608: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng18{k11=0} for conv (f32[240,1,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[14,240,56,56]{3,2,1,0}, f32[14,240,56,56]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=240, custom_call_target="__cudnn$convBackwardFilter", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule"

21/21 ━━━━━━━━━━━━━━━━━━━━ 189s 4s/step - accuracy: 0.5634 - loss: 1.1153 - val_accuracy: 0.2250 - val_loss: 1.2773
Epoch 2/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.5883 - loss: 1.1176 - val_accuracy: 0.2250 - val_loss: 216.4315
Epoch 3/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 235ms/step - accuracy: 0.7950 - loss: 0.5083 - val_accuracy: 0.7750 - val_loss: 1186.4011
Epoch 4/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 237ms/step - accuracy: 0.8843 - loss: 0.4824 - val_accuracy: 0.2250 - val_loss: 7.4330
Epoch 5/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 311ms/step - accuracy: 0.9337 - loss: 0.2196 - val_accuracy: 0.2250 - val_loss: 12.8719
Epoch 6/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 239ms/step - accuracy: 0.9211 - loss: 0.2157 - val_accuracy: 0.2250 - val_loss: 117.7212
Epoch 7/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 238ms/step - accuracy: 0.9773 - loss: 0.0855 - val_accuracy: 0.7750 - val_loss: 2.2554
Epoch 8/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 237ms/step - accuracy: 0.9285 - loss: 0.2261 - val_accuracy: 0.2250

## Accuracy

In [12]:
model.load_weights('../weights/EfficientNet.weights.h5')
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

2025-06-02 08:39:10.089264: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5552', 16 bytes spill stores, 16 bytes spill loads

2025-06-02 08:39:10.505449: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5552_0', 112 bytes spill stores, 252 bytes spill loads

2025-06-02 08:39:10.572494: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5552_0', 244 bytes spill stores, 244 bytes spill loads

2025-06-02 08:39:10.611843: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5552', 228 bytes spill stores, 228 bytes spill loads



2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8672 - loss: 0.3165

2025-06-02 08:39:23.217043: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5552_0', 244 bytes spill stores, 244 bytes spill loads

2025-06-02 08:39:25.142902: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng4{} for conv (f32[16,32,112,112]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,56,112,112]{3,2,1,0}, f32[32,56,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-06-02 08:39:25.148962: E external/local_xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.255569504s
Trying algorithm eng4{} for conv (f32[16,32,112,112]{3,2,1,0}, u8[0]{0}) cust

3/3 ━━━━━━━━━━━━━━━━━━━━ 27s 5s/step - accuracy: 0.8648 - loss: 0.3359
Test Loss: 0.35525158047676086
Test Accuracy: 0.862500011920929
